# SkateFormer — Ewaluacja PKU-MMD na wszystkich wagach

Ewaluuje `PKU_XSub_both.npz` i `PKU_XView_both.npz` na wszystkich 8 zestawach wag.
Tam gdzie jest `SkateFormer_b.pt` liczy też ensemble j+b. Zadanie badane eksperymentalnie ma rozstrzygnąć, które wagi NTU RGB+D 60 są kanoniczne dla ewaluacji zero-shot na PKU-MMD (rozdz. 4.2–4.3 pracy). Porównanie ntu60_CSub vs ntu60_CView pokazuje, że wagi ntu60_CSub dają wynik kanoniczny (64,65% XSub / 64,67% XView), podczas gdy ntu60_CView daje wynik niższy i odrzucony jako niekanoniczny (60,65% XSub / 62,62% XView). Notebook zawiera też porównanie z wagami NTU120 oraz wariantami inter (ntu60_inter, ntu120_inter) jako dodatkowy kontekst.


## Struktura na Google Drive:
```
MyDrive/
  SkateFormer_weights/
    NTU60_CSub/          ← SkateFormer_j.pt  (+ opcjonalnie SkateFormer_b.pt)
    NTU60_CView/
    NTU60_inter_CSub/
    NTU60_inter_CView/
    NTU120_CSub/
    NTU120_CSet/
    NTU120_inter_CSub/
    NTU120_inter_CSet/
  PKU_data/
    PKU_XSub_both.npz
    PKU_XView_both.npz
```

In [ ]:
DRIVE_WEIGHTS_BASE = '/content/drive/MyDrive/SkateFormer_weights'
DRIVE_DATA_DIR     = '/content/drive/MyDrive/PKU_data'

WEIGHT_CONFIGS = [
    ('ntu60_CSub',        60),
    ('ntu60_CView',       60),
    ('ntu60_inter_CSub',  11),
    ('ntu60_inter_CView', 11),
    ('ntu120_CSub',      120),
    ('ntu120_CSet',      120),
    ('ntu120_inter_CSub', 26),
    ('ntu120_inter_CSet', 26),
]

PKU_DATASETS = [
    'PKU_XSub_both.npz',
    'PKU_XView_both.npz',
]
INTER_TO_NTU = {
    'ntu60_inter_CSub':  list(range(49, 60)),
    'ntu60_inter_CView': list(range(49, 60)),
    'ntu120_inter_CSub': list(range(49, 60)) + list(range(105, 120)),
    'ntu120_inter_CSet': list(range(49, 60)) + list(range(105, 120)),
}

def compute_top1_inter(score_pkl, npz_path, folder):
    #Dla modeli inter ewaluacja tylko na próbkach interakcyjnych
    inter_to_ntu = np.array(INTER_TO_NTU[folder])

    with open(score_pkl, 'rb') as f:
        scores = pickle.load(f)
    keys   = sorted(scores.keys(), key=lambda x: int(x.split('_')[1]))
    arr    = np.array([scores[k] for k in keys])

    data   = np.load(npz_path)
    y_true = np.argmax(data['y_test'], axis=1)

    valid_ntu = set(inter_to_ntu.tolist())
    mask      = np.array([y in valid_ntu for y in y_true])

    if mask.sum() == 0:
        print(f'Brak próbek interakcyjnych w tym zbiorze.')
        return None, arr

    arr_masked    = arr[mask]
    y_true_masked = y_true[mask]

    y_pred_inter = np.argmax(arr_masked, axis=1)
    y_pred_ntu   = inter_to_ntu[y_pred_inter]

    acc = (y_true_masked == y_pred_ntu).mean() * 100
    print(f'    (ewaluacja na {mask.sum()} próbkach interakcyjnych z {len(y_true)})')
    return acc, arr

print('Konfiguracja załadowana.')

Konfiguracja załadowana.


In [ ]:
!git clone https://github.com/KAIST-VICLab/SkateFormer.git
import os
os.chdir('/content/SkateFormer')
!ls

Cloning into 'SkateFormer'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 180 (delta 52), reused 43 (delta 35), pack-reused 101 (from 1)
Receiving objects: 100% (180/180), 1.12 MiB | 27.88 MiB/s, done.
Resolving deltas: 100% (80/80), done.
assets	data	 LICENSE  model		  README.md	     skateformer
config	feeders  main.py  pyproject.toml  requirements.yaml  torchlight


In [ ]:
!pip install -q einops==0.6.1 timm==0.9.12 tensorpack torchpack loguru msgpack msgpack-numpy tabulate tensorboardX

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.3/296.3 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, sys
os.chdir('/content/SkateFormer')

#Poprawka torchlight
with open('/content/SkateFormer/torchlight/torchlight/__init__.py', 'w') as f:
    f.write('''import argparse

class DictAction(argparse.Action):
    def __call__(self, parser, namespace, values, option_string=None):
        input_dict = getattr(namespace, self.dest, {}) or {}
        for kv in values:
            key, val = kv.split("=", 1)
            try:
                val = eval(val)
            except:
                pass
            input_dict[key] = val
        setattr(namespace, self.dest, input_dict)

class IO:
    pass
''')
print('Zastosowano poprawkę torchlight.')

#Usunięcie konfliktu z zainstalowaną wersją torchlight
os.system('pip uninstall torchlight -y')
if 'torchlight' in sys.modules:
    del sys.modules['torchlight']
sys.path.insert(0, '/content/SkateFormer/torchlight')
print('Usunięto konflikt z systemowym torchlight.')

#Poprawka main.py
with open('/content/SkateFormer/main.py', 'r') as f:
    content = f.read()
content = content.replace(
    'from torchlight import DictAction',
    '''class DictAction(argparse.Action):
    def __call__(self, parser, namespace, values, option_string=None):
        input_dict = getattr(namespace, self.dest, {}) or {}
        for kv in values:
            key, val = kv.split("=", 1)
            try:
                val = eval(val)
            except:
                pass
            input_dict[key] = val
        setattr(namespace, self.dest, input_dict)'''
)
content = content.replace(
    'default_arg = yaml.load(f)',
    'default_arg = yaml.load(f, Loader=yaml.SafeLoader)'
)
with open('/content/SkateFormer/main.py', 'w') as f:
    f.write(content)
print('Zastosowano poprawkę main.py.')

#Zapis co 5 epok
with open('/content/SkateFormer/main.py', 'r') as f:
    content = f.read()
old_code = """                if epoch + 1 < self.arg.num_epoch * 0.9:
                    self.train(epoch, save_model=False)
                else:
                    self.train(epoch, save_model=True)
                    self.eval(epoch, save_score=True, loader_name=['test'])"""
new_code = """                if epoch + 1 < self.arg.num_epoch * 0.9:
                    save = ((epoch + 1) % 5 == 0)
                    self.train(epoch, save_model=save)
                else:
                    self.train(epoch, save_model=True)
                    self.eval(epoch, save_score=True, loader_name=['test'])"""
if old_code in content:
    content = content.replace(old_code, new_code)
    with open('/content/SkateFormer/main.py', 'w') as f:
        f.write(content)
    print('Zapisuje co 5 epoki.')
elif '% 5 == 0' in content:
    print('Już zastosowane')
else:
    print('Niepowiodło się.')

#Zastąpienie przestarzałego np.int
with open('/content/SkateFormer/feeders/feeder_ntu.py', 'r') as f:
    content = f.read()
content = content.replace('.astype(np.int)', '.astype(int)')
with open('/content/SkateFormer/feeders/feeder_ntu.py', 'w') as f:
    f.write(content)
print('Zastąpiono przestarzały typ np.int.')

#Zastąpienie przestarzałych typów NumPy
with open('/content/SkateFormer/feeders/tools.py', 'r') as f:
    content = f.read()
for old, new in [('np.int)', 'int)'), ('np.int,', 'int,'),
                 ('np.float)', 'float)'), ('np.float,', 'float,'),
                 ('np.bool)', 'bool)'), ('np.complex)', 'complex)')]:
    content = content.replace(old, new)
with open('/content/SkateFormer/feeders/tools.py', 'w') as f:
    f.write(content)
print('Zastąpiono przestarzałe typy NumPy.')

print('\nZakończono dostosowanie kodu SkateFormer.')

Zastosowano poprawkę torchlight.
Usunięto konflikt z systemowym torchlight.
Zastosowano poprawkę main.py.
Zapisuje co 5 epoki.
Zastąpiono przestarzały typ np.int.
Zastąpiono przestarzałe typy NumPy.

Zakończono dostosowanie kodu SkateFormer.


In [ ]:
import shutil

os.makedirs('/content/SkateFormer/data/ntu', exist_ok=True)

for npz_name in PKU_DATASETS:
    src = f'{DRIVE_DATA_DIR}/{npz_name}'
    dst = f'/content/SkateFormer/data/ntu/{npz_name}'
    if os.path.exists(dst):
        print(f'Już istnieje: {npz_name}')
    elif os.path.exists(src):
        shutil.copy(src, dst)
        size = os.path.getsize(dst) / 1024**2
        print(f'Skopiowano: {npz_name}  ({size:.1f} MB)')
    else:
        print(f'Brak pliku na Drive: {src}')

import numpy as np
for npz_name in PKU_DATASETS:
    path = f'/content/SkateFormer/data/ntu/{npz_name}'
    if os.path.exists(path):
        d = np.load(path)
        print(f'  {npz_name}: train={d["x_train"].shape}, test={d["x_test"].shape}')

Skopiowano: PKU_XSub_both.npz  (693.9 MB)
Skopiowano: PKU_XView_both.npz  (693.9 MB)
  PKU_XSub_both.npz: train=(18713, 300, 150), test=(2696, 300, 150)
  PKU_XView_both.npz: train=(14268, 300, 150), test=(7141, 300, 150)


In [ ]:
WEIGHTS_LOCAL_BASE = '/content/SkateFormer/weights'
os.makedirs(WEIGHTS_LOCAL_BASE, exist_ok=True)

available_weights = {}

for folder, num_classes in WEIGHT_CONFIGS:
    local_dir = f'{WEIGHTS_LOCAL_BASE}/{folder}'
    os.makedirs(local_dir, exist_ok=True)
    available_weights[folder] = {'j': None, 'b': None, 'num_classes': num_classes}

    for pt_name, key in [('SkateFormer_j.pt', 'j'), ('SkateFormer_b.pt', 'b')]:
        src = f'{DRIVE_WEIGHTS_BASE}/{folder}/{pt_name}'
        dst = f'{local_dir}/{pt_name}'
        if os.path.exists(dst):
            available_weights[folder][key] = dst
        elif os.path.exists(src):
            shutil.copy(src, dst)
            available_weights[folder][key] = dst
            size = os.path.getsize(dst) / 1024**2
            print(f'{folder}/{pt_name}  ({size:.1f} MB)')
        else:
            if key == 'j':
                print(f'BRAK: {folder}/{pt_name}')
            else:
                print(f'Brak: {folder}/{pt_name}  (ensemble niedostępny)')

print('\nPodsumowanie wag:')
for folder, info in available_weights.items():
    j = 'jest' if info['j'] else 'nie ma'
    b = 'jest' if info['b'] else 'niedostępny'
    print(f'  {folder:25s}  _j:{j}  _b:{b}  classes:{info["num_classes"]}')

ntu60_CSub/SkateFormer_j.pt  (13.9 MB)
ntu60_CSub/SkateFormer_b.pt  (13.9 MB)
ntu60_CView/SkateFormer_j.pt  (13.9 MB)
ntu60_CView/SkateFormer_b.pt  (13.9 MB)
ntu60_inter_CSub/SkateFormer_j.pt  (13.8 MB)
Brak: ntu60_inter_CSub/SkateFormer_b.pt  (ensemble niedostępny)
ntu60_inter_CView/SkateFormer_j.pt  (13.8 MB)
Brak: ntu60_inter_CView/SkateFormer_b.pt  (ensemble niedostępny)
ntu120_CSub/SkateFormer_j.pt  (13.9 MB)
ntu120_CSub/SkateFormer_b.pt  (13.9 MB)
ntu120_CSet/SkateFormer_j.pt  (13.9 MB)
ntu120_CSet/SkateFormer_b.pt  (13.9 MB)
ntu120_inter_CSub/SkateFormer_j.pt  (13.9 MB)
Brak: ntu120_inter_CSub/SkateFormer_b.pt  (ensemble niedostępny)
ntu120_inter_CSet/SkateFormer_j.pt  (13.9 MB)
Brak: ntu120_inter_CSet/SkateFormer_b.pt  (ensemble niedostępny)

Podsumowanie wag:
  ntu60_CSub                 _j:jest  _b:jest  classes:60
  ntu60_CView                _j:jest  _b:jest  classes:60
  ntu60_inter_CSub           _j:jest  _b:niedostępny  classes:11
  ntu60_inter_CView          _j:jest  _b

In [ ]:
import os

base = '/content/drive/MyDrive/SkateFormer_weights'
for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

SkateFormer_weights/
  .DS_Store
  ntu120_inter_CSet/
    SkateFormer_j.pt
  ntu60_CView/
    SkateFormer_j.pt
    SkateFormer_b.pt
  ntu120_CSet/
    SkateFormer_j.pt
    SkateFormer_b.pt
  ntu120_CSub/
    SkateFormer_j.pt
    SkateFormer_b.pt
  NWUCLA/
    SkateFormer_j.pt
    SkateFormer_b.pt
  ntu60_CSub/
    SkateFormer_j.pt
    SkateFormer_b.pt
  ntu120_inter_CSub/
    SkateFormer_j.pt
  ntu60_inter_CView/
    SkateFormer_j.pt
  ntu60_inter_CSub/
    SkateFormer_j.pt


In [ ]:
def fix_pocket_mapping(npz_path):
    data = dict(np.load(npz_path))
    total_fixed = 0
    for split in ['train', 'test']:
        y_idx = np.argmax(data[f'y_{split}'], axis=1)
        mask = y_idx == 56
        if mask.any():
            y_idx[mask] = 24
            new_onehot = np.zeros_like(data[f'y_{split}'])
            new_onehot[np.arange(len(y_idx)), y_idx] = 1.0
            data[f'y_{split}'] = new_onehot
            total_fixed += mask.sum()

    fixed_name = os.path.basename(npz_path).replace('.npz', '_fixed.npz')
    drive_fixed_path = f'{DRIVE_DATA_DIR}/{fixed_name}'
    local_fixed_path = f'/content/SkateFormer/data/ntu/{fixed_name}'

    np.savez(drive_fixed_path, **data)
    shutil.copy(drive_fixed_path, local_fixed_path)

    print(f'Naprawiono: {fixed_name} ({total_fixed} etykiet 56→24)')
    return fixed_name

PKU_DATASETS_FIXED = [fix_pocket_mapping(f'{DRIVE_DATA_DIR}/{name}') for name in PKU_DATASETS]
PKU_DATASETS = PKU_DATASETS_FIXED

PKU_DATASETS = PKU_DATASETS_FIXED

Naprawiono: PKU_XSub_both_fixed.npz (923 etykiet 56→24)
Naprawiono: PKU_XView_both_fixed.npz (923 etykiet 56→24)


In [ ]:
import yaml, pickle, subprocess

def build_config(weights_pt, npz_path, num_classes, work_dir, pt_key='j'):
    return {
        'seed': 1,
        'num_worker': 4,
        'work_dir': work_dir,
        'phase': 'test',
        'weights': weights_pt,
        'feeder': 'feeders.feeder_ntu.Feeder',
        'train_feeder_args': {
            'data_path': npz_path,
            'split': 'train',
            'debug': False,
            'data_type': 'b' if pt_key == 'b' else 'joint',
            'window_size': 64,
            'p_interval': [0.5, 1],
            'aug_method': 'a123489',
            'intra_p': 0.5,
            'inter_p': 0.2,
            'thres': 64,
            'uniform': True,
            'partition': True
        },
        'test_feeder_args': {
            'data_path': npz_path,
            'split': 'test',
            'data_type': 'b' if pt_key == 'b' else 'joint',
            'window_size': 64,
            'p_interval': [0.95],
            'thres': 64,
            'uniform': True,
            'partition': True,
            'debug': False
        },
        'model': 'model.SkateFormer.SkateFormer_',
        'model_args': {
            'num_classes': num_classes,
            'num_people': 2,
            'num_points': 24,
            'kernel_size': 7,
            'num_heads': 32,
            'attn_drop': 0.5,
            'head_drop': 0.0,
            'rel': True,
            'drop_path': 0.2,
            'type_1_size': [8, 8],
            'type_2_size': [8, 12],
            'type_3_size': [8, 8],
            'type_4_size': [8, 12],
            'mlp_ratio': 4.0,
            'index_t': True
        },
        'device': [0],
        'batch_size': 128,
        'test_batch_size': 128,
        'num_epoch': 1,
        'save_score': True
    }

def run_eval(weights_pt, npz_path, num_classes, work_dir, config_path, pt_key='j'):
    os.makedirs(os.path.dirname(config_path), exist_ok=True)
    os.makedirs(work_dir, exist_ok=True)

    cfg = build_config(weights_pt, npz_path, num_classes, work_dir, pt_key)
    with open(config_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)

    score_pkl = os.path.join(work_dir, 'epoch1_test_score.pkl')

    if os.path.exists(score_pkl):
        print(f'Wynik już istnieje: {score_pkl}')
        return score_pkl

    print(f'Uruchamiam: {config_path}')
    result = subprocess.run(
        ['python', 'main.py', '--config', config_path],
        capture_output=False
    )

    if result.returncode != 0:
        print(f'Błąd (returncode={result.returncode})')
        return None

    return score_pkl if os.path.exists(score_pkl) else None


def compute_top1(score_pkl, npz_path, num_classes=60):
    with open(score_pkl, 'rb') as f:
        scores = pickle.load(f)
    keys     = sorted(scores.keys(), key=lambda x: int(x.split('_')[1]))
    arr      = np.array([scores[k] for k in keys])
    arr_eval = arr[:, :60] if num_classes == 120 else arr
    data     = np.load(npz_path)
    y_true   = np.argmax(data['y_test'], axis=1)
    y_pred   = np.argmax(arr_eval, axis=1)
    return (y_true == y_pred).mean() * 100, arr


print('Funkcje pomocnicze zdefiniowane.')

Funkcje pomocnicze zdefiniowane.


In [ ]:
INTER_NTU_CLASSES = {
    'ntu60_inter':  list(range(49, 60)),
    'ntu120_inter': list(range(49, 60)),
}

inter_npz_cache = {}

for npz_name in PKU_DATASETS:
    npz_local = f'/content/SkateFormer/data/ntu/{npz_name}'
    npz_stem  = npz_name.replace('.npz', '')
    data      = np.load(npz_local)

    for prefix, ntu_indices in INTER_NTU_CLASSES.items():
        ntu_indices = np.array(ntu_indices)
        num_inter   = len(ntu_indices)
        out_path    = f'/content/SkateFormer/data/ntu/{npz_stem}_{prefix}.npz'
        inter_npz_cache[(prefix, npz_stem)] = out_path

        if os.path.exists(out_path):
            print(f'Już istnieje: {os.path.basename(out_path)}')
            continue

        for split in ['train', 'test']:
            x    = data[f'x_{split}']
            y_oh = data[f'y_{split}']
            y_idx = np.argmax(y_oh, axis=1)

            mask    = np.isin(y_idx, ntu_indices)
            x_filt  = x[mask]
            y_filt  = y_idx[mask]

            ntu_to_inter = {ntu: i for i, ntu in enumerate(ntu_indices)}
            y_remapped   = np.array([ntu_to_inter[y] for y in y_filt])

            y_new = np.zeros((len(y_remapped), num_inter), dtype=np.float32)
            y_new[np.arange(len(y_remapped)), y_remapped] = 1.0

            if split == 'train':
                x_train_out, y_train_out = x_filt, y_new
            else:
                x_test_out, y_test_out   = x_filt, y_new

        np.savez_compressed(out_path,
            x_train=x_train_out, y_train=y_train_out,
            x_test=x_test_out,   y_test=y_test_out)
        d = np.load(out_path)
        print(f'{os.path.basename(out_path)}: train={d["x_train"].shape}, test={d["x_test"].shape}')

PKU_XSub_both_fixed_ntu60_inter.npz: train=(1518, 300, 150), test=(195, 300, 150)
PKU_XSub_both_fixed_ntu120_inter.npz: train=(1518, 300, 150), test=(195, 300, 150)
PKU_XView_both_fixed_ntu60_inter.npz: train=(1142, 300, 150), test=(571, 300, 150)
PKU_XView_both_fixed_ntu120_inter.npz: train=(1142, 300, 150), test=(571, 300, 150)


In [ ]:
results = {}
score_store = {}

for folder, num_classes in WEIGHT_CONFIGS:
    info = available_weights[folder]
    if info['j'] is None:
        print(f'[{folder}] brak _j.pt')
        continue

    results[folder] = {}

    for npz_name in PKU_DATASETS:
        npz_stem   = npz_name.replace('.npz', '')
        if 'inter' in folder:
            prefix    = 'ntu60_inter' if '60' in folder else 'ntu120_inter'
            npz_local = inter_npz_cache[(prefix, npz_stem)]
        else:
            npz_local = f'/content/SkateFormer/data/ntu/{npz_name}'
        split_tag  = 'xsub' if 'XSub' in npz_name else 'xview'

        if not os.path.exists(npz_local):
            print(f'[{folder}] Brak danych: {npz_name}')
            continue

        print(f'\n[{folder}]  {npz_name}')
        results[folder][npz_stem] = {}

        for pt_key in ['j', 'b']:
            weights_pt = info[pt_key]
            if weights_pt is None:
                continue

            work_dir    = f'./work_dir/{folder}/{split_tag}/SkateFormer_{pt_key}'
            config_path = f'./config/eval/{folder}/{split_tag}_SkateFormer_{pt_key}.yaml'

            print(f'  [{pt_key}] weights={weights_pt}')
            score_pkl = run_eval(weights_pt, npz_local, num_classes, work_dir, config_path, pt_key)

            if score_pkl:
                acc, arr = compute_top1(score_pkl, npz_local, num_classes=num_classes)
                results[folder][npz_stem][pt_key] = acc
                score_store[(folder, npz_stem, pt_key)] = arr
                print(f'Top-1 ({pt_key}): {acc:.2f}%')

        key_j = (folder, npz_stem, 'j')
        key_b = (folder, npz_stem, 'b')
        if key_j in score_store and key_b in score_store:
            arr_j    = score_store[key_j]
            arr_b    = score_store[key_b]
            ensemble = (arr_j + arr_b) / 2
            ens_eval = ensemble[:, :60] if num_classes == 120 else ensemble
            data     = np.load(npz_local)
            y_true   = np.argmax(data['y_test'], axis=1)
            y_pred   = np.argmax(ens_eval, axis=1)
            acc_ens  = (y_true == y_pred).mean() * 100
            results[folder][npz_stem]['ensemble'] = acc_ens
            print(f'    → Top-1 (j+b ensemble): {acc_ens:.2f}%')

print('\n\nWszystkie ewaluacje zakończone.')


[ntu60_CSub]  PKU_XSub_both_fixed.npz
  [j] weights=/content/SkateFormer/weights/ntu60_CSub/SkateFormer_j.pt
Uruchamiam: ./config/eval/ntu60_CSub/xsub_SkateFormer_j.yaml
Top-1 (j): 64.65%
  [b] weights=/content/SkateFormer/weights/ntu60_CSub/SkateFormer_b.pt
Uruchamiam: ./config/eval/ntu60_CSub/xsub_SkateFormer_b.yaml
Top-1 (b): 74.93%
    → Top-1 (j+b ensemble): 73.55%

[ntu60_CSub]  PKU_XView_both_fixed.npz
  [j] weights=/content/SkateFormer/weights/ntu60_CSub/SkateFormer_j.pt
Uruchamiam: ./config/eval/ntu60_CSub/xview_SkateFormer_j.yaml
Top-1 (j): 64.67%
  [b] weights=/content/SkateFormer/weights/ntu60_CSub/SkateFormer_b.pt
Uruchamiam: ./config/eval/ntu60_CSub/xview_SkateFormer_b.yaml
Top-1 (b): 74.35%
    → Top-1 (j+b ensemble): 73.21%

[ntu60_CView]  PKU_XSub_both_fixed.npz
  [j] weights=/content/SkateFormer/weights/ntu60_CView/SkateFormer_j.pt
Uruchamiam: ./config/eval/ntu60_CView/xsub_SkateFormer_j.yaml
Top-1 (j): 60.65%
  [b] weights=/content/SkateFormer/weights/ntu60_CView/Sk

In [ ]:
import pandas as pd

rows = []
for folder, npz_dict in results.items():
    num_classes = available_weights[folder]['num_classes']
    for npz_stem, acc_dict in npz_dict.items():
        split = 'XSub' if 'XSub' in npz_stem else 'XView'
        row = {
            'Weights folder':  folder,
            'num_classes':     num_classes,
            'PKU split':       split,
            'Top-1 _j (%)':    round(acc_dict.get('j', float('nan')), 2),
            'Top-1 _b (%)':    round(acc_dict.get('b', float('nan')), 2),
            'Top-1 ens (%)':   round(acc_dict.get('ensemble', float('nan')), 2),
        }
        rows.append(row)

df = pd.DataFrame(rows)
df = df.sort_values(['PKU split', 'Weights folder']).reset_index(drop=True)

print(df.to_string(index=False))

df.to_csv('/content/drive/MyDrive/PKU_eval_results.csv', index=False)
print('\n Zapisano wyniki: MyDrive/PKU_eval_results.csv')

   Weights folder  num_classes PKU split  Top-1 _j (%)  Top-1 _b (%)  Top-1 ens (%)
      ntu120_CSet          120      XSub         54.41         71.51          69.99
      ntu120_CSub          120      XSub         60.83         72.14          72.89
ntu120_inter_CSet           26      XSub         76.92           NaN            NaN
ntu120_inter_CSub           26      XSub         80.00           NaN            NaN
       ntu60_CSub           60      XSub         64.65         74.93          73.55
      ntu60_CView           60      XSub         60.65         73.89          72.14
 ntu60_inter_CSub           11      XSub         81.03           NaN            NaN
ntu60_inter_CView           11      XSub         85.13           NaN            NaN
      ntu120_CSet          120     XView         57.25         73.25          71.32
      ntu120_CSub          120     XView         61.49         73.16          73.20
ntu120_inter_CSet           26     XView         80.04           NaN        